# Exploración de pedidos PDF recibidos por WhatsApp

## Objetivo
Validar la lectura y extracción del formato estándar de pedidos
antes de implementar su carga incremental en Bronze.

## Origen
abfss://receipts@graxianodatalake.dfs.core.windows.net/documents/

## Validación inicial
Usaremos el documento 2829651824098248.pdf para comprobar:
- Acceso desde Databricks.
- Extracción de cliente, fechas y observaciones.
- Extracción de las 10 líneas de productos.
- Conservación literal de las cantidades solicitadas.

No se modificarán los PDF ni se escribirán tablas en esta exploración.

In [0]:
documents_path = (
    "abfss://receipts@graxianodatalake.dfs.core.windows.net/"
    "documents/"
)

display(dbutils.fs.ls(documents_path))

In [0]:
display(spark.sql("SHOW EXTERNAL LOCATIONS"))

In [0]:
%sql DESCRIBE STORAGE CREDENTIAL cred_graxiano_adls

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ext_whatsapp_receipts
URL 'abfss://receipts@graxianodatalake.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL cred_graxiano_adls)
COMMENT 'Originales de WhatsApp: eventos y documentos adjuntos';

In [0]:
documents_path = (
    "abfss://receipts@graxianodatalake.dfs.core.windows.net/"
    "documents/"
)

display(dbutils.fs.ls(documents_path))

In [0]:
%pip install pdfplumber

In [0]:
from io import BytesIO

import pdfplumber

pdf_path = documents_path + "2829651824098248.pdf"

# Leemos únicamente el documento de prueba.
pdf_row = (
    spark.read
    .format("binaryFile")
    .load(pdf_path)
    .select("content")
    .first()
)

if pdf_row is None:
    raise RuntimeError("No se encontró el PDF.")

with pdfplumber.open(BytesIO(bytes(pdf_row.content))) as pdf:
    print("Páginas:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        print(f"\n--- Página {page_number} ---")
        print(page.extract_text(layout=False) or "[Sin texto extraíble]")

In [0]:
with pdfplumber.open(BytesIO(bytes(pdf_row.content))) as pdf:
    page = pdf.pages[0]
    words = page.extract_words()

    # La cabecera marca el inicio de la columna de cantidades.
    quantity_header = next(
        word for word in words
        if word["text"] == "Cantidad"
    )

    observations_header = next(
        word for word in words
        if word["text"] == "Observaciones"
    )

    column_boundary = quantity_header["x0"] - 5

    # Seleccionar únicamente las palabras del cuerpo de la tabla.
    body_words = [
        word for word in words
        if word["top"] > quantity_header["bottom"]
        and word["bottom"] < observations_header["top"]
    ]

    # Agrupar palabras que pertenecen a la misma línea.
    grouped_rows = []

    for word in sorted(body_words, key=lambda w: (w["top"], w["x0"])):
        if (
            not grouped_rows
            or abs(word["top"] - grouped_rows[-1][0]["top"]) > 3
        ):
            grouped_rows.append([word])
        else:
            grouped_rows[-1].append(word)

    order_lines = []

    for row in grouped_rows:
        row = sorted(row, key=lambda w: w["x0"])

        left = [
            word["text"] for word in row
            if word["x0"] < column_boundary
        ]
        right = [
            word["text"] for word in row
            if word["x0"] >= column_boundary
        ]

        if not left or not left[0].isdigit() or not right:
            raise ValueError("Se encontró una fila con formato inesperado.")

        order_lines.append({
            "line_number": int(left[0]),
            "product_raw": " ".join(left[1:]),
            "quantity_raw": " ".join(right),
        })

display(spark.createDataFrame(order_lines))

In [0]:
import re
from datetime import datetime

with pdfplumber.open(BytesIO(bytes(pdf_row.content))) as pdf:
    text = "\n".join(
        page.extract_text() or ""
        for page in pdf.pages
    )


def required_field(pattern, field_name):
    match = re.search(pattern, text, flags=re.MULTILINE)

    if not match:
        raise ValueError(f"No se encontró el campo: {field_name}")

    return match.group(1).strip()


customer_raw = required_field(
    r"^Cliente:\s*(.+)$",
    "cliente",
)

order_date_raw = required_field(
    r"^Fecha del pedido:\s*(\d{2}/\d{2}/\d{4})\s*$",
    "fecha del pedido",
)

delivery_date_raw = required_field(
    r"^Entrega solicitada:\s*(\d{2}/\d{2}/\d{4})\s*$",
    "entrega solicitada",
)

# Validar el formato sin modificar los valores originales.
for date_value in (order_date_raw, delivery_date_raw):
    datetime.strptime(date_value, "%d/%m/%Y")

notes_match = re.search(
    r"^Observaciones de entrega\s*\n"
    r"(.*?)"
    r"(?=^DOCUMENTO DE EJEMPLO|\Z)",
    text,
    flags=re.MULTILINE | re.DOTALL,
)

header = {
    "customer_raw": customer_raw,
    "order_date_raw": order_date_raw,
    "delivery_date_raw": delivery_date_raw,
    "delivery_notes_raw": (
        notes_match.group(1).strip()
        if notes_match else None
    ),
}

display(spark.createDataFrame([header]))

In [0]:
import importlib
import sys
from pathlib import Path

current = Path.cwd()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "src/orders").is_dir()
        and (path / "databricks.yml").is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

import orders.pdf_parser as parser

importlib.reload(parser)

order = parser.extract_order(bytes(pdf_row.content))

print("Versión del extractor:", order["parser_version"])
display(spark.createDataFrame([order["header"]]))
display(spark.createDataFrame(order["lines"]))

## Conclusión de la prueba

El extractor `orders.pdf_parser`, versión 1.0.0, recupera correctamente
la cabecera y las 10 líneas del documento de ejemplo.

Las cantidades se conservan literalmente y no se requiere OCR.
La prueba valida únicamente la plantilla de una página analizada.

No se han escrito todavía tablas Bronze.